# MCP (Model Context Protocol) Server Implementation

In [1]:
!python -m pip install fastmcp langchain_mcp_adapters --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
os.makedirs("mcpserver",exist_ok=True)

In [3]:
%%writefile mcpserver/mcpserver1.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
async def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
async def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='streamable-http')


Writing mcpserver/mcpserver1.py


In [9]:
# run mCP server: python mcpserver/mcpserver1.py

## Implement an Agent which connects to the MCP server and fetches the tools

In [1]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [6]:
client = MultiServerMCPClient({"EYMCP":{"url":"http://127.0.0.1:8000/mcp",
                                              "transport":"streamable_http"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description='this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000015BA9C8FF60>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000015BA9C39F80>)]

In [7]:
agent = create_agent("azure_openai:gpt-4o-mini",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='e34d3ef7-c46f-4e0b-83bb-0e209c9b2d1d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 73, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DIC0sGDrPdBmlXoEI3UmFTksQ0Lae', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity

In [8]:
await agent.ainvoke({"messages":[{"role":'user',"content":"Tell me more about city Jaiselmer?"}]})

{'messages': [HumanMessage(content='Tell me more about city Jaiselmer?', additional_kwargs={}, response_metadata={}, id='107af633-6626-4105-be53-4c8943fc9bf6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 76, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DIC1aPBRRgVZPmF98Odv6lVwe2xjp', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'sev

In [9]:
%%writefile mcpserver/mcpserver2.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='stdio')


Writing mcpserver/mcpserver2.py


In [2]:
client = MultiServerMCPClient({"EYMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()
tools

UnsupportedOperation: fileno

In [ ]:
agent = create_agent("azure_openai:gpt-4o-mini",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='b46a3baa-3661-4788-901b-9700c2469f66'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--12135834-e63e-4fc7-a34c-9ff4d364021e-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '7d594495-55c3-4bee-9681-8612ff405062', 'type': 'tool_call'}], usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='{\n  "city": "Delhi",\n  "weather": "haze",\n  "temperature": 291.2,\n  "unit": "kelvin"\n}', name='get_current_weather', id='63573841-f811-4ac3-afd6-ea7e7bfdae56', tool_call_id='7d59